###Model Finetuning with LoRA

Llama 3.2

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
!git clone https://github.com/DillonR03/Potential-Talents-Candidate-Ranking-Model.git
%cd Potential-Talents-Candidate-Ranking-Model

Cloning into 'Potential-Talents-Candidate-Ranking-Model'...
remote: Enumerating objects: 15135, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15135 (delta 1), reused 10 (delta 1), pack-reused 15123 (from 2)
Receiving objects: 100% (15135/15135), 110.07 MiB | 16.18 MiB/s, done.
Resolving deltas: 100% (1517/1517), done.
Updating files: 100% (14803/14803), done.
/content/Potential-Talents-Candidate-Ranking-Model


In [ ]:
!ls

data		    Fine_Tuning_Colab_Notebook.ipynb  README.md
final_merged_model  PotentialTalentsNotebook.ipynb


In [ ]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attemptin

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully


In [ ]:
import pandas as pd

df = pd.read_excel(
    "data/Extended Dataset for Potential Talents 7.xlsx"
)

print(df.shape)
df.head()

(1285, 4)


,id,title,location,screening_score
0,1,innovative and driven professional seeking a r...,United States,100
1,2,ms applied data science student usc research a...,United States,100
2,3,computer science student seeking full-time sof...,United States,100
3,4,microsoft certified power bi data analyst mba ...,United States,100
4,5,graduate research assistant at uab masters in ...,United States,100


In [ ]:
pd.set_option("display.max_colwidth", 200)

display(df.head(10))

,id,title,location,screening_score
0,1,innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.,United States,100
1,2,ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025,United States,100
2,3,computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs,United States,100
3,4,microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-eri...,United States,100
4,5,graduate research assistant at uab masters in data science student at uab ex jio,United States,100
5,6,student at kennesaw state university,United States,100
6,7,data analyst business analyst python snowflake sql machine learning power bi tableau equipped with analytics driven by insights and passionate about impactful solutions.,United States,100
7,8,graduate research aide student at arizona state university,United States,100
8,9,data science nlp ai ml python sas quantum computing block chain sql,United States,100
9,10,data science machine learning artificial intelligence,United States,100


In [ ]:
for column in df.columns:
    print(f"\n===== {column} =====")
    print(df[column].head(5).tolist())


===== id =====
[1, 2, 3, 4, 5]

===== title =====
['innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.', 'ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025', 'computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs', 'microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson', 'graduate research assistant at uab masters in data science student at uab ex jio']

===== location =====
['United States', 'United States', 'United States', 'United States', 'United States']

===== screening_score =====
[100, 100, 100, 100, 100]


In [ ]:
SYSTEM_PROMPT = """You are an expert recruitment analyst.

Your task is to evaluate how well a candidate's title matches a given recruitment search query.
I will provide you with a target search query and a candidate's title.

Evaluate the candidates title using these four criteria.

1. ROLE RELEVANCE (0-25)
How closely is the candidate's current or stated role related to the target field?

2. CAREER INTENT (0-25)
Does the candidate's title indicate that they are seeking, aspiring toward,
studying, or otherwise interested in the target field?

3. PROFESSIONAL EXPERIENCE (0-25)
Does the candidate demonstrate relevant professional experience in the target field?

4. OVERALL FIT (0-25)
Considering the candidate's title as a whole, how suitable does the candidate
appear for the recruitment search?

SCORING GUIDELINES:

0-5   = No meaningful relevance
6-10  = Weak relevance
11-15 = Moderate relevance
16-20 = Strong relevance
21-25 = Very strong relevance

IMPORTANT:
- Base your assessment ONLY on the candidate's job title.
- Do not assume information that is not present.
- Distinguish between someone who is already working in the field and someone
  who merely expresses an interest in entering the field.
- A direct HR role should generally score higher than a role with only an
  indirect connection to HR.
- An "aspiring" or "seeking" HR candidate should receive credit for career
  intent.
- Do not give every candidate the same score."""

In [ ]:
df["text"] = df.apply(
    lambda x: f"{SYSTEM_PROMPT}\n### Human: Given the target search query: aspiring human resources, and the candidate's title: {x['title']}. What is the screening score? ### Assistant: {x['screening_score']}",
    axis=1,
)
print("Text column created with system prompt, search query, and title-based screening format.")

Text column created with system prompt, search query, and title-based screening format.


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(f"Dataset created: {dataset}")

Dataset created: Dataset({
    features: ['id', 'title', 'location', 'screening_score', 'text'],
    num_rows: 1285
})


In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Dataset split into train and test: {dataset}")

Dataset split into train and test: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 1028
    })
    test: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 257
    })
})


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization config
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Set tokenizer padding token to be the same as the EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

print("Base model and tokenizer loaded with quantization.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Base model and tokenizer loaded with quantization.


Next, we'll set up the LoRA configuration to enable efficient fine-tuning of the model.

In [ ]:
from peft import LoraConfig

# LoRA configuration
lora_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA configuration set.")

LoRA configuration set.


Now, we define the training arguments for our `SFTTrainer`.

In [ ]:
from transformers import TrainingArguments

# Training arguments
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    # warmup_ratio=0.03, # Removed as it causes TypeError
    # group_by_length=True, # Removed as it causes TypeError
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

print("Training arguments defined.")

Training arguments defined.


Finally, we initialize the `SFTTrainer` and start the fine-tuning process.

In [ ]:
from trl import SFTTrainer

# Initialize SFTTrainer
sft_trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    # dataset_text_field="text", # Removed as it causes TypeError
    # tokenizer=tokenizer, # Removed as it causes TypeError
    args=training_arguments,
    # packing=False, # Removed as it causes TypeError
    # max_seq_length=1024 # Removed as it causes TypeError
)

# Train the model
sft_trainer.train()

print("Model training started.")

Adding EOS to train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
100,0.551332
200,0.170640


Model training started.


In [ ]:
# Evaluate the model
metrics = sft_trainer.evaluate()
print("Evaluation Metrics:", metrics)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.170470,0.171350,257,0.212699,374023.000000,0.970056


Evaluation Metrics: {'eval_loss': 0.1713496446609497, 'eval_entropy': 0.21269923358252554, 'eval_num_tokens': 374023.0, 'eval_mean_token_accuracy': 0.9700563983483748}


In [ ]:
# Save the fine-tuned model
output_merged_dir = "./results/final_merged_model"
sft_trainer.model.save_pretrained(output_merged_dir)
tokenizer.save_pretrained(output_merged_dir)
print(f"Fine-tuned model and tokenizer saved to {output_merged_dir}")

Fine-tuned model and tokenizer saved to ./results/final_merged_model


In [ ]:
import shutil
from google.colab import files

# Define the path to the folder to be zipped
folder_to_zip = './results'
output_zip_file = 'results.zip'

# Create a zip archive of the folder
shutil.make_archive(output_zip_file.replace('.zip', ''), 'zip', folder_to_zip)

print(f"Folder '{folder_to_zip}' has been zipped to '{output_zip_file}'")

Folder './results' has been zipped to 'results.zip'


In [ ]:
# Download the zipped file
files.download(output_zip_file)
print(f"Download initiated for '{output_zip_file}'")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated for 'results.zip'


Model Performance Evaluation

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded")
print("Device:", model.device)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Base model loaded
Device: cuda:0


In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from peft import PeftModel

adapter_path = output_merged_dir

model = PeftModel.from_pretrained(
    model,
    adapter_path,
    offload_dir="/tmp/" # Add an offload directory for layers moved to CPU
)

model.eval()

print("Fine-tuned LoRA adapter loaded from local path:", adapter_path)

Fine-tuned LoRA adapter loaded from local path: ./results/final_merged_model


In [ ]:
model.print_trainable_parameters()

trainable params: 0 || all params: 3,231,099,904 || trainable%: 0.0000


In [ ]:
import pandas as pd
import os

# Change to the cloned repository's directory to correctly resolve relative paths
repo_root_abs_path = "/content/Potential-Talents-Candidate-Ranking-Model"
os.chdir(repo_root_abs_path)

# Confirm current working directory after change
current_dir = os.getcwd()
print(f"Current Working Directory: {current_dir}")

# Assuming 'data' is a subdirectory of the current working directory
data_directory_relative_path = "data"

# Diagnostic checks for the relative path
print(f"Checking path existence for: {os.path.join(current_dir, data_directory_relative_path)}")
if not os.path.exists(data_directory_relative_path):
    print(f"Error: Directory does not exist at {data_directory_relative_path} relative to current directory.")
elif not os.path.isdir(data_directory_relative_path):
    print(f"Error: Path exists but is not a directory: {data_directory_relative_path}")
else:
    print(f"Directory exists and is accessible: {data_directory_relative_path}")

# List the contents of the 'data' directory using its relative path
print(f"Contents of {data_directory_relative_path}:", os.listdir(data_directory_relative_path))

# Construct the full relative path for the candidate file
candidate_file_name = "Extended Dataset for Potential Talents 7.xlsx"
candidate_path = os.path.join(data_directory_relative_path, candidate_file_name)

print(f"Attempting to read from: {candidate_path}")

candidates = pd.read_excel(candidate_path)

print("Candidates:", candidates.shape)
candidates.head()

Current Working Directory: /content/Potential-Talents-Candidate-Ranking-Model
Checking path existence for: /content/Potential-Talents-Candidate-Ranking-Model/data
Directory exists and is accessible: data
Contents of data: ['potential-talents - Aspiring human resources - seeking human resources.csv', 'Extended Dataset for Potential Talents 7.xlsx']
Attempting to read from: data/Extended Dataset for Potential Talents 7.xlsx
Candidates: (1285, 4)


,id,title,location,screening_score
0,1,innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.,United States,100
1,2,ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025,United States,100
2,3,computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs,United States,100
3,4,microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-eri...,United States,100
4,5,graduate research assistant at uab masters in data science student at uab ex jio,United States,100


In [ ]:
search_query = "machine learning"

print("Search query:", search_query)

Search query: machine learning


In [ ]:
def create_ranking_prompt(candidate_title, search_query):
    # Use the globally defined SYSTEM_PROMPT and append the specific query in the Human/Assistant format.
    return f"{SYSTEM_PROMPT}\n### Human: Given the target search query: {search_query}, and the candidate's title: {candidate_title}. What is the screening score? ### Assistant:"


In [ ]:
test_title = candidates.iloc[0]["title"]

prompt = create_ranking_prompt(
    test_title,
    search_query
)

print(prompt)

You are an expert recruitment analyst.

Your task is to evaluate how well a candidate's title matches a given recruitment search query.
I will provide you with a target search query and a candidate's title.

Evaluate the candidates title using these four criteria.

1. ROLE RELEVANCE (0-25)
How closely is the candidate's current or stated role related to the target field?

2. CAREER INTENT (0-25)
Does the candidate's title indicate that they are seeking, aspiring toward,
studying, or otherwise interested in the target field?

3. PROFESSIONAL EXPERIENCE (0-25)
Does the candidate demonstrate relevant professional experience in the target field?

4. OVERALL FIT (0-25)
Considering the candidate's title as a whole, how suitable does the candidate
appear for the recruitment search?

SCORING GUIDELINES:

0-5   = No meaningful relevance
6-10  = Weak relevance
11-15 = Moderate relevance
16-20 = Strong relevance
21-25 = Very strong relevance

IMPORTANT:
- Base your assessment ONLY on the candidat

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        temperature=None
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Candidate:")
print(test_title)

print("\nModel response:")
print(response)

Candidate:
innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.

Model response:
 95


In [ ]:
ranking_scores = []

for index, row in candidates.iterrows():
    candidate_title = row["title"]
    prompt = create_ranking_prompt(candidate_title, search_query)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=None
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        score = int(response)
    except ValueError:
        score = 0 # Default to 0 if the model doesn't return a valid integer score

    ranking_scores.append(score)

candidates["predicted_score"] = ranking_scores

# Rank candidates by the predicted score in descending order
ranked_candidates = candidates.sort_values(by="predicted_score", ascending=False)

print("Candidates ranked by predicted screening score:")
display(ranked_candidates.head(10))
display(ranked_candidates.tail(10))

Candidates ranked by predicted screening score:


,id,title,location,screening_score,predicted_score
1281,1282,Data Scientist and Analyst Driving Business Insights with Advanced Data Techniques Research Expertise,Kenya,0,95
0,1,innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.,United States,100,95
1242,1243,Data Engineer Data Analyst ex-ThoughtSpot Solution Analyst Skilled in SQL Python dbt Snowflake Airflow ETL Data Modeling Analytics Cloud Data Engineering,India,0,95
1244,1245,AIML Developer Abdul Kalam Research Award Recipient,Nepal,0,95
1247,1248,Machine Learning Engineer Google Certified Associate Cloud Engineer M-Tech Silver Medalist in Artificial Intelligence Data Science.,India,0,95
1248,1249,B.Sc. Hons in Data Science Python AI ML LLM NLP Data Analysis,Sri Lanka,0,95
1249,1250,PYTHON SQL POWER BI ADVANCE EXCEL MACHINE LEARNING TABLEAU,India,0,95
1234,1235,Data Scientist with Expertise in Machine LearningVisualizationPredictive Modeling,India,0,95
1235,1236,Data Scientist Data Analyst Python SQL ML AI developer GenerativeAI Kaggle Expert Linux Cloud GPUTPU Model Training NLP Pharm.d,Pakistan,0,95
1236,1237,Machine Learning Engineer Data Scientist MLOps,Nigeria,0,95


,id,title,location,screening_score,predicted_score
714,715,--,United States,80,0
719,720,deli associate at kroger,United States,80,0
384,385,--,United States,100,0
818,819,--,United States,80,0
431,432,--,United States,95,0
880,881,--,United States,70,0
884,885,--,United States,70,0
891,892,--,United States,70,0
892,893,--,United States,70,0
1274,1275,Student,Russia,0,0


Qwen

## Fine-tuning Qwen/Qwen2.5-3B-Instruct Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# New model ID for Qwen
model_id = "Qwen/Qwen2.5-3B-Instruct"

# Load the tokenizer for the new model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization config
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Set tokenizer padding token to be the same as the EOS token for Qwen
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

print("New base model (Qwen) and tokenizer loaded with quantization.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

New base model (Qwen) and tokenizer loaded with quantization.


In [ ]:
from peft import LoraConfig

# LoRA configuration (same as before)
lora_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA configuration re-applied.")

LoRA configuration re-applied.


Next, we'll define the training arguments for our `SFTTrainer` for the Qwen model.

In [ ]:
from transformers import TrainingArguments

# Training arguments (same as before)
training_arguments = TrainingArguments(
    output_dir="./qwen_results", # Changed output directory to avoid conflict
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

print("Training arguments defined for Qwen.")

Training arguments defined for Qwen.


In [ ]:
import pandas as pd
import os

# Ensure we are in the correct directory if needed (e.g., if you cloned a repo)
# For robustness, we can set the working directory to where the Excel file is expected
# Based on previous execution, the repo is cloned in /content/Potential-Talents-Candidate-Ranking-Model
# and the data is in /content/Potential-Talents-Candidate-Ranking-Model/data
repo_root_abs_path = "/content/Potential-Talents-Candidate-Ranking-Model"
if os.getcwd() != repo_root_abs_path:
    os.chdir(repo_root_abs_path)

# Construct the full path for the Excel file
data_directory_relative_path = "data"
candidate_file_name = "Extended Dataset for Potential Talents 7.xlsx"
candidate_path = os.path.join(data_directory_relative_path, candidate_file_name)

# Re-load the DataFrame
df = pd.read_excel(candidate_path)

print("DataFrame 'df' re-loaded.")

DataFrame 'df' re-loaded.


In [ ]:
# Re-define the SYSTEM_PROMPT (if it was cleared)
SYSTEM_PROMPT = """You are an expert recruitment analyst.

Your task is to evaluate how well a candidate's title matches a given recruitment search query.
I will provide you with a target search query and a candidate's title.

Evaluate the candidates title using these four criteria.

1. ROLE RELEVANCE (0-25)
How closely is the candidate's current or stated role related to the target field?

2. CAREER INTENT (0-25)
Does the candidate's title indicate that they are seeking, aspiring toward,
studying, or otherwise interested in the target field?

3. PROFESSIONAL EXPERIENCE (0-25)
Does the candidate demonstrate relevant professional experience in the target field?

4. OVERALL FIT (0-25)
Considering the candidate's title as a whole, how suitable does the candidate
appear for the recruitment search?

SCORING GUIDELINES:

0-5   = No meaningful relevance
6-10  = Weak relevance
11-15 = Moderate relevance
16-20 = Strong relevance
21-25 = Very strong relevance

IMPORTANT:
- Base your assessment ONLY on the candidate's job title.
- Do not assume information that is not present.
- Distinguish between someone who is already working in the field and someone
  who merely expresses an interest in entering the field.
- A direct HR role should generally score higher than a role with only an
  indirect connection to HR.
- An "aspiring" or "seeking" HR candidate should receive credit for career
  intent.
- Do not give every candidate the same score."""

# Re-create the 'text' column in the DataFrame
df["text"] = df.apply(
    lambda x: f"{SYSTEM_PROMPT}\n### Human: Given the target search query: data science, and the candidate's title: {x['title']}. What is the screening score? ### Assistant: {x['screening_score']}",
    axis=1,
)
print("Text column re-created in 'df'.")

Text column re-created in 'df'.


In [ ]:
from datasets import Dataset

# Create Dataset from the re-loaded DataFrame
dataset = Dataset.from_pandas(df)

print(f"Dataset created for Qwen: {dataset}")

Dataset created for Qwen: Dataset({
    features: ['id', 'title', 'location', 'screening_score', 'text'],
    num_rows: 1285
})


In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Dataset split into train and test for Qwen: {dataset}")

Dataset split into train and test for Qwen: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 1028
    })
    test: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 257
    })
})


Finally, we initialize the `SFTTrainer` and start the fine-tuning process for the Qwen model.

In [ ]:
from trl import SFTTrainer

# Initialize SFTTrainer for Qwen
sft_trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    args=training_arguments
)

# Train the model
sft_trainer.train()

print("Qwen model training started.")

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 17.81 MiB is free. Including non-PyTorch memory, this process has 14.46 GiB memory in use. Of the allocated memory 14.21 GiB is allocated by PyTorch, and 118.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Evaluate the Qwen model
metrics = sft_trainer.evaluate()
print("Evaluation Metrics for Qwen:", metrics)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.145048,0.145968,257,0.153131,385467.000000,0.972556


Evaluation Metrics for Qwen: {'eval_loss': 0.14596815407276154, 'eval_entropy': 0.1531313540357532, 'eval_num_tokens': 385467.0, 'eval_mean_token_accuracy': 0.9725559064836213}


In [ ]:
# Save the fine-tuned Qwen model
output_merged_dir_qwen = "./qwen_results/final_merged_qwen_model"
sft_trainer.model.save_pretrained(output_merged_dir_qwen)
tokenizer.save_pretrained(output_merged_dir_qwen)
print(f"Fine-tuned Qwen model and tokenizer saved to {output_merged_dir_qwen}")

Fine-tuned Qwen model and tokenizer saved to ./qwen_results/final_merged_qwen_model


Qwen Model Performance Evaluation

### Load Fine-Tuned Qwen Model for Evaluation

In [ ]:
!pip install --upgrade torchao

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Ensure torchao is updated to a compatible version before PeftModel is used
!pip install --upgrade torchao

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

qwen_base_model_id = "Qwen/Qwen2.5-3B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_base_model_id)
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token # Ensure padding token is set
qwen_tokenizer.padding_side = "right"

qwen_base_model = AutoModelForCausalLM.from_pretrained(
    qwen_base_model_id,
    dtype=torch.float16,
    device_map="auto"
)

# Load the fine-tuned adapter for Qwen
qwen_adapter_path = output_merged_dir_qwen # This variable was defined earlier
qwen_model = PeftModel.from_pretrained(
    qwen_base_model,
    qwen_adapter_path,
    offload_dir="/tmp/qwen_offload" # Use a distinct offload directory
)

qwen_model.eval()

print("Fine-tuned Qwen model loaded for evaluation.")
print("Device:", qwen_model.device)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Fine-tuned Qwen model loaded for evaluation.
Device: cuda:0


In [ ]:
qwen_model.print_trainable_parameters()

trainable params: 0 || all params: 3,100,684,288 || trainable%: 0.0000


In [ ]:
import pandas as pd
import os

# Ensure we are in the correct directory if needed (e.g., if you cloned a repo)
repo_root_abs_path = "/content/Potential-Talents-Candidate-Ranking-Model"
if os.getcwd() != repo_root_abs_path:
    os.chdir(repo_root_abs_path)

# Construct the full path for the Excel file
data_directory_relative_path = "data"
candidate_file_name = "Extended Dataset for Potential Talents 7.xlsx"
candidate_path = os.path.join(data_directory_relative_path, candidate_file_name)

# Re-load the candidates DataFrame
candidates = pd.read_excel(candidate_path)

print("Candidates DataFrame re-loaded for Qwen evaluation.")

Candidates DataFrame re-loaded for Qwen evaluation.


In [ ]:
search_query = "datascience"

print("Search query for Qwen evaluation:", search_query)

Search query for Qwen evaluation: datascience


In [ ]:
def create_ranking_prompt(candidate_title, search_query):
    # Use the globally defined SYSTEM_PROMPT and append the specific query in the Human/Assistant format.
    return f"{SYSTEM_PROMPT}\n### Human: Given the target search query: {search_query}, and the candidate's title: {candidate_title}. What is the screening score? ### Assistant:"

test_title_qwen = candidates.iloc[0]["title"]

prompt_qwen = create_ranking_prompt(
    test_title_qwen,
    search_query
)

inputs_qwen = qwen_tokenizer(
    prompt_qwen,
    return_tensors="pt"
).to(qwen_model.device)

with torch.no_grad():
    outputs_qwen = qwen_model.generate(
        **inputs_qwen,
        max_new_tokens=10,
        do_sample=False,
        temperature=None
    )

response_qwen = qwen_tokenizer.decode(
    outputs_qwen[0][inputs_qwen["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Candidate:")
print(test_title_qwen)

print("\nQwen Model response:")
print(response_qwen)

Candidate:
innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.

Qwen Model response:
 85


### Rank All Candidates with Fine-Tuned Qwen Model

In [ ]:
qwen_ranking_scores = []

for index, row in candidates.iterrows():
    candidate_title = row["title"]
    prompt = create_ranking_prompt(candidate_title, search_query)

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qwen_model.device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=None
        )

    response = qwen_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        score = int(response)
    except ValueError:
        score = 0 # Default to 0 if the model doesn't return a valid integer score

    qwen_ranking_scores.append(score)

candidates["predicted_score_qwen"] = qwen_ranking_scores

# Rank candidates by the predicted score in descending order
ranked_candidates_qwen = candidates.sort_values(by="predicted_score_qwen", ascending=False)

print("Candidates ranked by Qwen predicted screening score:")
display(ranked_candidates_qwen.head(10))
display(ranked_candidates_qwen.tail(10))

Candidates ranked by Qwen predicted screening score:


,id,title,location,screening_score,predicted_score_qwen
205,206,aiml researcher data scientist aws certified a...,United States,35,95
602,603,ms student in ai long island university certif...,United States,85,95
495,496,data analyst sales operation analytics compens...,United States,95,95
1262,1263,Data Analyst MLOps Engineer Orchestrating Infr...,India,0,95
212,213,data scientist specializing in machine learnin...,United States,35,95
815,816,software engineer full stack developer javascr...,United States,80,85
814,815,data analyst hca healthcare masters in compute...,United States,80,85
813,814,python developer,United States,80,85
862,863,data analyst business intelligence analyst,Brazil,75,85
861,862,student at university of cincinnati,United States,75,85


,id,title,location,screening_score,predicted_score_qwen
388,389,machine learning researcher data scientist tra...,United States,100,0
224,225,seeking full-time roles digital transformation...,United States,35,0
99,100,aspiring data analyst business intelligence sq...,United States,85,0
133,134,data analyst turning data into actionable insi...,United States,80,0
49,50,senior data analyst at iqvia working with phar...,United States,95,0
54,55,ai enthusiast graduate assistant at kent state...,United States,95,0
67,68,new grad 24 data analyst seeking new opportuni...,United States,90,0
65,66,ms in biotechnology ms candidate in biological...,United States,90,0
1235,1236,Data Scientist Data Analyst Python SQL ML AI d...,Pakistan,0,0
1239,1240,Deep Learning NLP Image Processing Data Scienc...,India,0,0
